# Apache Spark Physical Plan Demo

This notebook demonstrates:
- **File Formats**: CSV, JSON, Parquet comparison
- **Physical Plan Components**: Scan, Exchange, HashAggregate, BroadcastHashJoin, SortMergeJoin
- **Performance Optimization**: Caching, broadcast joins

**Dataset**: Iris dataset (small, ~5KB) + synthetic sales data

---
## 1. Setup Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, broadcast, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import time
import os

# Create Spark Session
spark = SparkSession.builder \
    .appName("PhysicalPlanDemo") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 07:52:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.1.1
Spark UI: http://192.168.0.150:4040


---
## 2. Load Public Dataset (Iris - ~5KB)

We'll download the Iris dataset from UCI Machine Learning Repository.

In [3]:
import requests
import shutil
import warnings
from urllib3.exceptions import InsecureRequestWarning

# Suppress only the single InsecureRequestWarning from urllib3 needed for this download
warnings.simplefilter('ignore', InsecureRequestWarning)

# Create data directory
os.makedirs("data", exist_ok=True)

# Download Iris dataset from UCI ML Repository using requests
iris_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

try:
    # Use requests to get the file, bypassing SSL verification
    response = requests.get(iris_url, stream=True, verify=False)
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    # Save the content to a file
    with open("data/iris.csv", "wb") as out_file:
        shutil.copyfileobj(response.raw, out_file)

    print("Downloaded Iris dataset")
    print(f"File size: {os.path.getsize('data/iris.csv') / 1024:.2f} KB")

except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")

Downloaded Iris dataset
File size: 4.44 KB


In [4]:
# Define schema for Iris dataset
iris_schema = StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width", DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width", DoubleType(), True),
    StructField("species", StringType(), True)
])

In [5]:
# Load CSV
iris_df = spark.read.csv("data/iris.csv", schema=iris_schema)

print(f"Total records: {iris_df.count()}")
iris_df.show(5)

Total records: 150
+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|    species|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


---
## 3. Create Additional DataFrames for Joins

We'll create lookup tables and a sales dataset to demonstrate different join types.

In [6]:
# Species lookup table (SMALL - will be broadcast)
species_info = spark.createDataFrame([
    ("Iris-setosa", "Setosa", "Small", 10.99),
    ("Iris-versicolor", "Versicolor", "Medium", 15.99),
    ("Iris-virginica", "Virginica", "Large", 19.99)
], ["species", "common_name", "size_category", "price"])

print("Species Info (Small Lookup Table):")
species_info.show()

Species Info (Small Lookup Table):


+---------------+-----------+-------------+-----+
|        species|common_name|size_category|price|
+---------------+-----------+-------------+-----+
|    Iris-setosa|     Setosa|        Small|10.99|
|Iris-versicolor| Versicolor|       Medium|15.99|
| Iris-virginica|  Virginica|        Large|19.99|
+---------------+-----------+-------------+-----+



In [7]:
# Region lookup table (SMALL - will be broadcast)
regions = spark.createDataFrame([
    ("Small", "North America", 0.0),
    ("Medium", "Europe", 5.0),
    ("Large", "Asia", 8.0)
], ["size_category", "region", "shipping_cost"])

print("Regions Lookup Table:")
regions.show()

Regions Lookup Table:
+-------------+-------------+-------------+
|size_category|       region|shipping_cost|
+-------------+-------------+-------------+
|        Small|North America|          0.0|
|       Medium|       Europe|          5.0|
|        Large|         Asia|          8.0|
+-------------+-------------+-------------+



In [8]:
# Create synthetic orders table (larger dataset)
import random
random.seed(42)

species_list = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
orders_data = [
    (i, random.choice(species_list), random.randint(1, 10), f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}")
    for i in range(1, 1001)  # 1000 orders
]

orders = spark.createDataFrame(orders_data, ["order_id", "species", "quantity", "order_date"])
print(f"Orders table: {orders.count()} records")
orders.show(5)

Orders table: 1000 records
+--------+---------------+--------+----------+
|order_id|        species|quantity|order_date|
+--------+---------------+--------+----------+
|       1| Iris-virginica|       2|2024-01-24|
|       2|Iris-versicolor|       4|2024-04-05|
|       3| Iris-virginica|       2|2024-11-24|
|       4| Iris-virginica|       2|2024-10-14|
|       5|    Iris-setosa|       1|2024-02-07|
+--------+---------------+--------+----------+
only showing top 5 rows


---
## 4. File Format Comparison: CSV vs JSON vs Parquet

Let's save and read data in different formats to compare performance.

In [9]:
# Save in different formats
orders.write.mode("overwrite").csv("data/orders_csv", header=True)
orders.write.mode("overwrite").json("data/orders_json")
orders.write.mode("overwrite").parquet("data/orders_parquet")

print("Saved orders in CSV, JSON, and Parquet formats")

26/05/21 07:53:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Saved orders in CSV, JSON, and Parquet formats


In [10]:
# Helper function to get folder size
def get_folder_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            total += os.path.getsize(fp)
    return total

In [11]:
# Compare file sizes
csv_size = get_folder_size("data/orders_csv")
json_size = get_folder_size("data/orders_json")
parquet_size = get_folder_size("data/orders_parquet")

print("File Size Comparison:")
print(f"  CSV:     {csv_size / 1024:.2f} KB")
print(f"  JSON:    {json_size / 1024:.2f} KB")
print(f"  Parquet: {parquet_size / 1024:.2f} KB")
print(f"\n  Parquet is {csv_size / parquet_size:.1f}x smaller than CSV!")

File Size Comparison:
  CSV:     31.28 KB
  JSON:    81.19 KB
  Parquet: 22.36 KB

  Parquet is 1.4x smaller than CSV!


In [12]:
# Compare read performance
def time_read(read_func):
    start = time.time()
    df = read_func()
    df.count()
    return time.time() - start

csv_time = time_read(lambda: spark.read.csv("data/orders_csv", header=True, inferSchema=True))
json_time = time_read(lambda: spark.read.json("data/orders_json"))
parquet_time = time_read(lambda: spark.read.parquet("data/orders_parquet"))

print("Read Performance:")
print(f"  CSV:     {csv_time:.3f} seconds")
print(f"  JSON:    {json_time:.3f} seconds")
print(f"  Parquet: {parquet_time:.3f} seconds")

Read Performance:
  CSV:     0.233 seconds
  JSON:    0.292 seconds
  Parquet: 0.163 seconds


### Parquet Benefits: Column Pruning

In [ ]:
# Column Pruning Demo
print("Column Pruning Demo:")
print("Parquet - Select only 'species' and 'quantity':")

parquet_pruned = spark.read.parquet("data/orders_parquet").select("species", "quantity")
parquet_pruned.explain()

print("\nNotice: ReadSchema only includes selected columns!")

In [ ]:
# Predicate Pushdown Demo
print("Predicate Pushdown Demo:")
print("Parquet - Filter quantity > 5:")

parquet_filtered = spark.read.parquet("data/orders_parquet").filter(col("quantity") > 5)
parquet_filtered.explain()

print("\nNotice: PushedFilters shows the filter is pushed to scan level!")

---
## 5. Physical Plan Components

### 5.1 SCAN - Reading Data

In [15]:
print("="*60)
print("1. SCAN - Reading Data from Source")
print("="*60)

1. SCAN - Reading Data from Source


In [ ]:
# Simple scan with filter
scan_df = spark.read.parquet("data/orders_parquet").filter(col("quantity") > 5)

print("Query: SELECT * FROM orders WHERE quantity > 5")
print("\nPhysical Plan:")
scan_df.explain(mode="formatted")

### 5.2 EXCHANGE + HASHAGGREGATION - Shuffle and Aggregation

In [20]:
print("="*60)
print("2. EXCHANGE + HASHAGGREGATION - Shuffle and Aggregation")
print("="*60)

2. EXCHANGE + HASHAGGREGATION - Shuffle and Aggregation


In [ ]:
# Aggregation query
agg_df = spark.read.parquet("data/orders_parquet") \
    .groupBy("species") \
    .agg(
        sum("quantity").alias("total_quantity"),
        count("*").alias("order_count"),
        spark_round(avg("quantity"), 2).alias("avg_quantity")
    )

print("Query: SELECT species, SUM(quantity), COUNT(*), AVG(quantity) FROM orders GROUP BY species")
print("\nPhysical Plan:")
agg_df.explain(mode="formatted")

In [22]:
print("Result:")
agg_df.show()

+---------------+--------------+-----------+------------+
|        species|total_quantity|order_count|avg_quantity|
+---------------+--------------+-----------+------------+
|    Iris-setosa|          1645|        301|        5.47|
| Iris-virginica|          2029|        369|         5.5|
|Iris-versicolor|          1825|        330|        5.53|
+---------------+--------------+-----------+------------+



### 5.3 BROADCASTHASHJOIN - Optimized Join for Small Tables

In [23]:
print("="*60)
print("3. BROADCASTHASHJOIN - Optimized Join for Small Tables")
print("="*60)

3. BROADCASTHASHJOIN - Optimized Join for Small Tables


In [24]:
# Load orders from parquet
orders_pq = spark.read.parquet("data/orders_parquet")

In [ ]:
# Join with small lookup table (will be broadcast)
broadcast_join_df = orders_pq.join(
    broadcast(species_info),
    "species"
)

print("Query: SELECT * FROM orders JOIN species_info ON species")
print("       (species_info is broadcast)")
print("\nPhysical Plan:")
broadcast_join_df.explain(mode="formatted")

In [26]:
print("Result (first 5 rows):")
broadcast_join_df.show(5)

+---------------+--------+--------+----------+-----------+-------------+-----+
|        species|order_id|quantity|order_date|common_name|size_category|price|
+---------------+--------+--------+----------+-----------+-------------+-----+
| Iris-virginica|       1|       2|2024-01-24|  Virginica|        Large|19.99|
|Iris-versicolor|       2|       4|2024-04-05| Versicolor|       Medium|15.99|
| Iris-virginica|       3|       2|2024-11-24|  Virginica|        Large|19.99|
| Iris-virginica|       4|       2|2024-10-14|  Virginica|        Large|19.99|
|    Iris-setosa|       5|       1|2024-02-07|     Setosa|        Small|10.99|
+---------------+--------+--------+----------+-----------+-------------+-----+
only showing top 5 rows



### 5.4 SORTMERGEJOIN - Join Requiring Shuffle & Sort

In [27]:
print("="*60)
print("4. SORTMERGEJOIN - Join Requiring Shuffle & Sort")
print("="*60)

4. SORTMERGEJOIN - Join Requiring Shuffle & Sort


In [28]:
# Disable broadcast to force SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
print("Broadcast disabled to force SortMergeJoin")

Broadcast disabled to force SortMergeJoin


In [ ]:
# Join two tables (forced SortMergeJoin)
sort_merge_df = orders_pq.join(species_info, "species")

print("Query: SELECT * FROM orders JOIN species_info ON species")
print("       (broadcast DISABLED)")
print("\nPhysical Plan:")
sort_merge_df.explain(mode="formatted")

In [30]:
# Reset broadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
print("Broadcast threshold reset to default (10MB)")

Broadcast threshold reset to default (10MB)


### 5.5 Complex Query - All Components Together

In [31]:
print("="*60)
print("5. COMPLEX QUERY - All Components Together")
print("="*60)

5. COMPLEX QUERY - All Components Together


In [32]:
# Disable broadcast for first join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [33]:
complex_df = spark.read.parquet("data/orders_parquet") \
    .filter(col("quantity") > 3) \
    .join(species_info, "species") \
    .join(broadcast(regions), "size_category") \
    .withColumn("total_price", col("quantity") * col("price") + col("shipping_cost")) \
    .groupBy("region", "common_name") \
    .agg(
        sum("quantity").alias("total_units"),
        spark_round(sum("total_price"), 2).alias("total_revenue"),
        count("*").alias("num_orders")
    ) \
    .orderBy(col("total_revenue").desc())

In [ ]:
print("Physical Plan:")
complex_df.explain(mode="formatted")

In [35]:
# Reset broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

In [36]:
print("Query Result:")
complex_df.show()

+-------------+-----------+-----------+-------------+----------+
|       region|common_name|total_units|total_revenue|num_orders|
+-------------+-----------+-----------+-------------+----------+
|         Asia|  Virginica|       1811|     38305.89|       263|
|       Europe| Versicolor|       1625|     27128.75|       229|
|North America|     Setosa|       1486|     16331.14|       220|
+-------------+-----------+-----------+-------------+----------+



---
## 6. Caching & Persistence Demo

In [37]:
print("="*60)
print("CACHING & PERSISTENCE DEMO")
print("="*60)

CACHING & PERSISTENCE DEMO


In [38]:
# Read data (no cache)
orders_no_cache = spark.read.parquet("data/orders_parquet")

In [39]:
# Without cache
start = time.time()
result1 = orders_no_cache.groupBy("species").count().collect()
result2 = orders_no_cache.groupBy("species").agg(sum("quantity")).collect()
no_cache_time = time.time() - start
print(f"Without cache (2 actions): {no_cache_time:.3f} seconds")

Without cache (2 actions): 0.334 seconds


In [40]:
# With cache
orders_cached = spark.read.parquet("data/orders_parquet").cache()
orders_cached.count()  # Materialize cache
print("Cache materialized")

Cache materialized


In [41]:
start = time.time()
result1 = orders_cached.groupBy("species").count().collect()
result2 = orders_cached.groupBy("species").agg(sum("quantity")).collect()
cache_time = time.time() - start
print(f"With cache (2 actions): {cache_time:.3f} seconds")

if cache_time > 0:
    print(f"Cache speedup: {no_cache_time / cache_time:.1f}x faster")

With cache (2 actions): 0.399 seconds
Cache speedup: 0.8x faster


In [42]:
# Storage levels
print("\nStorage Levels Available:")
print("  MEMORY_ONLY       - Fast, but may not fit")
print("  MEMORY_AND_DISK   - Recommended default")
print("  DISK_ONLY         - For very large data")
print("  MEMORY_ONLY_SER   - Serialized, saves memory")


Storage Levels Available:
  MEMORY_ONLY       - Fast, but may not fit
  MEMORY_AND_DISK   - Recommended default
  DISK_ONLY         - For very large data
  MEMORY_ONLY_SER   - Serialized, saves memory


In [43]:
# Unpersist
orders_cached.unpersist()
print("Cache cleared")

Cache cleared


---
## 7. Performance Optimization Tips Demo

### Broadcast Join vs Sort Merge Join

In [44]:
print("="*60)
print("BROADCAST JOIN vs SORT MERGE JOIN COMPARISON")
print("="*60)

BROADCAST JOIN vs SORT MERGE JOIN COMPARISON


In [45]:
orders_pq = spark.read.parquet("data/orders_parquet")

In [46]:
# Broadcast Join timing
start = time.time()
broadcast_result = orders_pq.join(broadcast(species_info), "species").count()
broadcast_time = time.time() - start
print(f"Broadcast Hash Join: {broadcast_time:.3f} seconds")

Broadcast Hash Join: 0.132 seconds


In [47]:
# Sort Merge Join timing
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
start = time.time()
smj_result = orders_pq.join(species_info, "species").count()
smj_time = time.time() - start
print(f"Sort Merge Join: {smj_time:.3f} seconds")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

if broadcast_time > 0:
    print(f"\nBroadcast is {smj_time / broadcast_time:.1f}x faster!")

Sort Merge Join: 0.241 seconds

Broadcast is 1.8x faster!


### Filter Early - Predicate Pushdown

In [48]:
print("="*60)
print("FILTER EARLY - PREDICATE PUSHDOWN")
print("="*60)

FILTER EARLY - PREDICATE PUSHDOWN


In [ ]:
# Bad: Filter after join
print("BAD: Filter AFTER join")
bad_df = orders_pq.join(broadcast(species_info), "species").filter(col("quantity") > 8)
bad_df.explain()

In [ ]:
# Good: Filter before join
print("\nGOOD: Filter BEFORE join")
good_df = orders_pq.filter(col("quantity") > 8).join(broadcast(species_info), "species")
good_df.explain()

---
## 8. Summary: Physical Plan Components

In [53]:
# Stop Spark session
spark.stop()
print("Spark session stopped")

Spark session stopped


In [54]:
# Optional: Clean up data files
import shutil
shutil.rmtree("data", ignore_errors=True)
print("Data files cleaned up")

Data files cleaned up
